In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('online_retail_clean.csv', parse_dates=['InvoiceDate'])
print(f"Wczytano: {df.shape}")
print(df.dtypes)

Wczytano: (801291, 9)
Invoice                 int64
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             int64
Country                   str
TotalPrice            float64
dtype: object


In [2]:
# Data referencyjna — dzień po ostatniej transakcji
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"Data referencyjna: {snapshot_date.date()}")

# Obliczenie cech RFM dla każdego klienta
rfm = df.groupby('Customer ID').agg(
    Recency   = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('Invoice', 'nunique'),
    Monetary  = ('TotalPrice', 'sum')
).reset_index()

print(f"\nRozmiar tabeli RFM: {rfm.shape}")
print(f"\nStatystyki opisowe:")
print(rfm[['Recency', 'Frequency', 'Monetary']].describe().round(2))

Data referencyjna: 2011-12-10



Rozmiar tabeli RFM: (5859, 4)

Statystyki opisowe:
       Recency  Frequency   Monetary
count  5859.00    5859.00    5859.00
mean    200.98       6.22    2912.83
std     209.15      12.69   14310.41
min       1.00       1.00       2.90
25%      26.00       1.00     343.56
50%      96.00       3.00     877.70
75%     379.00       7.00    2266.77
max     738.00     375.00  607539.25


In [3]:
# Cechy dodatkowe
rfm_extra = df.groupby('Customer ID').agg(
    ProductDiversity = ('StockCode', 'nunique'),
    AvgOrderValue    = ('TotalPrice', lambda x: x.sum() / df.loc[x.index, 'Invoice'].nunique()),
    PreferredHour    = ('InvoiceDate', lambda x: x.dt.hour.mode()[0]),
    PreferredDay     = ('InvoiceDate', lambda x: x.dt.dayofweek.mode()[0])
).reset_index()

rfm = rfm.merge(rfm_extra, on='Customer ID')

print(f"Cechy po rozszerzeniu: {rfm.shape}")
print(rfm.head())

Cechy po rozszerzeniu: (5859, 8)
   Customer ID  Recency  Frequency  Monetary  ProductDiversity  AvgOrderValue  \
0        12346      529         11    372.86                26      33.896364   
1        12347        2          8   5633.32               126     704.165000   
2        12348       75          5   1658.40                24     331.680000   
3        12349       19          3   3678.69               137    1226.230000   
4        12350      310          1    294.40                16     294.400000   

   PreferredHour  PreferredDay  
0             13             0  
1             14             1  
2             14             0  
3              9             3  
4             16             2  


In [4]:
print(rfm.columns.tolist())

['Customer ID', 'Recency', 'Frequency', 'Monetary', 'ProductDiversity', 'AvgOrderValue', 'PreferredHour', 'PreferredDay']


In [5]:
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Cechy do skalowania (bez Customer ID)
features = ['Recency', 'Frequency', 'Monetary', 
            'ProductDiversity', 'AvgOrderValue',
            'PreferredHour', 'PreferredDay']

# Transformacja logarytmiczna najpierw dla skośnych zmiennych
rfm_log = rfm.copy()
for col in ['Recency', 'Frequency', 'Monetary', 
            'ProductDiversity', 'AvgOrderValue']:
    rfm_log[col] = np.log1p(rfm_log[col])

# Standaryzacja
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log[features])
rfm_scaled = pd.DataFrame(rfm_scaled, columns=features)

print("Statystyki po skalowaniu (powinny być ~0 mean, ~1 std):")
print(rfm_scaled.describe().round(2))

Statystyki po skalowaniu (powinny być ~0 mean, ~1 std):
       Recency  Frequency  Monetary  ProductDiversity  AvgOrderValue  \
count  5859.00    5859.00   5859.00           5859.00        5859.00   
mean     -0.00       0.00      0.00              0.00          -0.00   
std       1.00       1.00      1.00              1.00           1.00   
min      -2.41      -1.06     -3.95             -2.53          -5.84   
25%      -0.74      -1.06     -0.71             -0.63          -0.59   
50%       0.08      -0.20     -0.03              0.05           0.03   
75%       0.95       0.66      0.66              0.73           0.56   
max       1.38       5.44      4.70              3.37           5.84   

       PreferredHour  PreferredDay  
count        5859.00       5859.00  
mean            0.00          0.00  
std             1.00          1.00  
min            -2.47         -1.37  
25%            -0.70         -0.84  
50%             0.19         -0.30  
75%             0.63          0.77  

In [6]:
# AvgDaysBetweenPurchases — średni czas między zakupami
avg_days = df.groupby('Customer ID').apply(
    lambda x: x.drop_duplicates('Invoice')
               .sort_values('InvoiceDate')['InvoiceDate']
               .diff()
               .dt.days
               .mean()
).reset_index()
avg_days.columns = ['Customer ID', 'AvgDaysBetweenPurchases']

# Klienci z tylko 1 fakturą dostaną NaN — wypełniamy Recency
rfm = rfm.merge(avg_days, on='Customer ID')
rfm['AvgDaysBetweenPurchases'] = rfm['AvgDaysBetweenPurchases'].fillna(rfm['Recency'])

print(rfm[['Customer ID', 'AvgDaysBetweenPurchases']].describe().round(2))

       Customer ID  AvgDaysBetweenPurchases
count      5859.00                  5859.00
mean      15316.28                   172.78
std        1715.82                   184.20
min       12346.00                     0.00
25%       13833.50                    45.33
50%       15317.00                    91.00
75%       16798.50                   236.00
max       18287.00                   738.00


In [7]:
features = ['Recency', 'Frequency', 'Monetary', 
            'ProductDiversity', 'AvgOrderValue',
            'PreferredHour', 'PreferredDay',
            'AvgDaysBetweenPurchases']

rfm_log = rfm.copy()
for col in ['Recency', 'Frequency', 'Monetary', 
            'ProductDiversity', 'AvgOrderValue',
            'AvgDaysBetweenPurchases']:
    rfm_log[col] = np.log1p(rfm_log[col])

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log[features])
rfm_scaled = pd.DataFrame(rfm_scaled, columns=features)

print("Cechy końcowe:", features)
print(f"Wymiar macierzy cech: {rfm_scaled.shape}")

# Zapis
rfm.to_csv('rfm_features.csv', index=False)
rfm_scaled.to_csv('rfm_scaled.csv', index=False)
print("\nZapisano pliki.")

Cechy końcowe: ['Recency', 'Frequency', 'Monetary', 'ProductDiversity', 'AvgOrderValue', 'PreferredHour', 'PreferredDay', 'AvgDaysBetweenPurchases']
Wymiar macierzy cech: (5859, 8)

Zapisano pliki.
